# Hangpost Matching Engine — Evaluation Comparison

This notebook reproduces the rules / rules+embeddings / learned comparison from
`scripts/train.py`, but with plots so the head-to-head is visible at a glance.

**Outputs are intentionally cleared** in version control. To populate plots:

```bash
pip install -e ".[ml]" pandas matplotlib seaborn jupyter
python scripts/train.py --with-embeddings   # produces models/learned_ranker.joblib
jupyter notebook notebooks/02_evaluation.ipynb
```

Cells flagged `[ml]` require the optional ML extras to run.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")

In [ ]:
from hangpost_matching import (
    build_queries,
    evaluate_ranker,
    load_profiles_from_csv,
    make_random_ranker,
    make_rules_ranker,
    split_queries,
)

profiles = load_profiles_from_csv(REPO_ROOT / "data" / "test_profiles.csv")
queries = [q for q in build_queries(profiles, n_sources=200, seed=42) if q[2]]
_, test_queries = split_queries(queries, train_fraction=0.7, seed=42)
print(f"{len(test_queries)} held-out test queries")

## Phase 1 baseline: random vs rules-only

These two run with no ML deps. The rules ranker should crush random — if it
doesn't, either the labels or the ranker have a bug.

In [ ]:
K_VALUES = [5, 10, 25, 50]

def _evaluate_at_ks(name, ranker, queries, ks):
    rows = []
    for k in ks:
        r = evaluate_ranker(ranker, queries, k=k)
        rows.append({"system": name, "k": k, "P@k": r.precision, "R@k": r.recall, "NDCG@k": r.ndcg, "MAP@k": r.map})
    return rows

rows = []
rows += _evaluate_at_ks("random", make_random_ranker(42), test_queries, K_VALUES)
rows += _evaluate_at_ks("rules_only", make_rules_ranker(), test_queries, K_VALUES)
phase12 = pd.DataFrame(rows)
phase12

## Phase 2: rules + sentence-transformer embeddings  `[ml]`

Loads `all-MiniLM-L6-v2` (~90MB on first run) and embeds every profile from
the auto-synthesized `profile_to_text` string. The rules ranker then has
access to a `semantic_similarity` feature.

In [ ]:
from hangpost_matching import SentenceTransformerEmbedder, embed_profiles

embedder = SentenceTransformerEmbedder()
embeddings = embed_profiles(profiles, embedder)
print(f"Embedded {len(embeddings)} profiles.")

rows += _evaluate_at_ks(
    "rules+embeddings",
    make_rules_ranker(profile_embeddings=embeddings),
    test_queries,
    K_VALUES,
)
df = pd.DataFrame(rows)
df.tail(len(K_VALUES) + 1)

## Phase 3: learned ranker  `[ml]`

Loads the LightGBM `LGBMRanker` saved by `scripts/train.py`.

In [ ]:
from hangpost_matching import LearnedRanker

model_path = REPO_ROOT / "models" / "learned_ranker.joblib"
if not model_path.exists():
    raise SystemExit(
        "Run `python scripts/train.py --with-embeddings` first to produce "
        f"{model_path}."
    )
learned = LearnedRanker.load(model_path)
rows += _evaluate_at_ks("learned", learned.as_ranker(), test_queries, K_VALUES)
df = pd.DataFrame(rows)
df.tail(len(K_VALUES))

## Side-by-side comparison

Bar chart of NDCG@k across all systems. The gap between rules_only and
rules+embeddings tells you what semantic similarity buys; the gap between
rules+embeddings and learned tells you what LightGBM learns on top.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for metric, ax in zip(["NDCG@k", "MAP@k"], axes):
    sns.barplot(data=df, x="k", y=metric, hue="system", ax=ax)
    ax.set_title(metric)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for metric, ax in zip(["P@k", "R@k"], axes):
    sns.barplot(data=df, x="k", y=metric, hue="system", ax=ax)
    ax.set_title(metric)
plt.tight_layout()
plt.show()

## Per-query NDCG@10 distribution  `[ml]`

Macro-averaged metrics hide variance. A system can win on average while
still losing on a long tail of queries — worth knowing.

In [ ]:
from hangpost_matching import ndcg_at_k

rules_ranker = make_rules_ranker()
embed_ranker = make_rules_ranker(profile_embeddings=embeddings)
learned_ranker = learned.as_ranker()

per_query = []
for source, candidates, relevant in test_queries:
    per_query.append({
        "rules_only": ndcg_at_k(rules_ranker(source, candidates), relevant, 10),
        "rules+embeddings": ndcg_at_k(embed_ranker(source, candidates), relevant, 10),
        "learned": ndcg_at_k(learned_ranker(source, candidates), relevant, 10),
    })
pq = pd.DataFrame(per_query).melt(var_name="system", value_name="NDCG@10")

fig, ax = plt.subplots(figsize=(8, 4))
sns.violinplot(data=pq, x="system", y="NDCG@10", inner="quartile", ax=ax)
ax.set_title("Per-query NDCG@10 distribution")
plt.tight_layout()
plt.show()

## Resume bullet template

After running this notebook, fill in the placeholders below and copy into the README and your CV:

> Designed a hybrid friend-recommendation system combining rule-based scoring,
> sentence-transformer semantic similarity, and a LightGBM LambdaRank model;
> evaluated with precision@k / recall@k / NDCG@k / MAP@k on a held-out
> {N} query split. Improved NDCG@10 from {X} (rules only) to {Y} (rules+embeddings)
> to {Z} (learned). MIT-licensed, CI on Python 3.10–3.12 with ruff/mypy/pytest.